# ESM-IF1 Inverse Folding

This notebook demonstrates both functions of the ESM-IF1 tool. In the first section, we use `run_esm_if1_sample` to generate new amino acid sequences conditioned on a target backbone structure. In the second section, we use `run_esm_if1_score` to evaluate how well a given sequence fits a target structure by computing average log-likelihood and perplexity.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm_if1")
display_overview("esm_if1")
display_docs_section("esm_if1", "Background")

## Available tools

In [2]:
display_available_tools("esm_if1")

### `run_esm_if1_sample`

Given a backbone structure, ESM-IF1 generates new amino acid sequences that are predicted to fold into the target conformation. You can choose between vanilla ESM-IF1 weights or the ProteinDPO variant, which is optimized for designing thermodynamically stable proteins. Sampling temperature controls the diversity of the generated sequences, with lower temperatures producing more conservative designs that closely match the structural constraints.

In [3]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1SampleConfig,
    ESMIF1SampleInput,
    run_esm_if1_sample,
)
from proto_tools.tools.inverse_folding.shared_data_models import (
    InverseFoldingInput,
    InverseFoldingStructureInput,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_sample")

# Load GFP as the target backbone
gfp_structure = get_gfp_structure()

struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    chain_ids=["A"],
)
inputs = InverseFoldingInput(inputs=[struct_input])

In [5]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_sample")

# Configure sampling | see docs above for all fields
config = ESMIF1SampleConfig(
    num_sequences_per_structure=3,
    temperature=0.1,
    weights_variant="protein_dpo",  # or "esmif" for vanilla ESM-IF1
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [6]:
# Run the sampling function
result = run_esm_if1_sample(inputs, config)

ESM-IF1 sampling:   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_sample")

# Print designed sequences and their log-likelihoods
for i, seq_res in enumerate(result.designed_sequences):
    for j, seq in enumerate(seq_res.sequences):
        ll = seq_res.log_likelihoods[j]
        display_seq = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"Structure {i}, Design {j + 1}: {display_seq}")
        print(f"  Length: {len(seq)} residues, Log-likelihood: {ll:.4f}")

Structure 0, Design 1: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -0.3543
Structure 0, Design 2: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -0.3543
Structure 0, Design 3: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -0.3592


### `run_esm_if1_score`

ESM-IF1 can evaluate how well a given amino acid sequence fits a target backbone structure by computing average log-likelihood and perplexity. This is useful for ranking candidate sequences from other design methods, comparing wild-type to mutant sequences, or validating that a designed sequence is structurally compatible with the intended fold. Scoring uses the full complex context, so multi-chain interactions are accounted for.

In [8]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1ScoringConfig,
    ESMIF1ScoringInput,
    run_esm_if1_score,
)
from proto_tools.tools.inverse_folding.shared_data_models import SequenceStructurePair

In [9]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_score")

# Score the first designed sequence against the GFP backbone
designed_seq = result.designed_sequences[0].sequences[0]

scoring_input = ESMIF1ScoringInput(
    sequence_structure_pairs=[
        SequenceStructurePair(
            sequence=designed_seq,
            structure=gfp_structure,
        )
    ]
)

In [10]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_score")

# Configure scoring | see docs above for all fields
scoring_config = ESMIF1ScoringConfig(
    weights_variant="protein_dpo",
    device="cuda",  # Change to "cpu" if no GPU available
)

In [11]:
# Run the scoring function
score_result = run_esm_if1_score(scoring_input, scoring_config)

ESM-IF1 scoring:   0%|          | 0/1 [00:00<?, ?pair/s]

In [12]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_score")

# Print scores for the designed sequence
print(f"Avg log-likelihood: {score_result.scores[0].avg_log_likelihood:.4f}")
print(f"Perplexity: {score_result.scores[0].perplexity:.4f}")

Avg log-likelihood: -0.3543
Perplexity: 1.4252


## Export Results

Sampling results can be exported to FASTA format for use in downstream sequence analysis pipelines. Scoring results can be exported to CSV or JSON format for further analysis or comparison.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="esm_if1_sequences", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'esm_if1_sequences.fasta'}")

score_result.export(name="esm_if1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'esm_if1_scores.csv'}")

Designed sequences exported to example_output/esm_if1_sequences.fasta
Scores exported to example_output/esm_if1_scores.csv
